# Análisis Exploratorio de Datos (EDA) — Taller 1

## Caso: Predicción de brotes de dengue

**Grupo 3**

---

Este cuaderno sigue la **misma estructura metodológica** de la guía del profesor (`EDA_Guia_Automoviles.ipynb`), adaptada al caso de **dengue** del equipo.

El flujo que seguiremos es:

> **Pregunta SMART + Hipótesis → Diccionario de datos → Visión general → Limpieza → Análisis univariado → Análisis bivariado → Conclusiones**

Documentación complementaria: [`docs/taller1/`](../docs/taller1/) · [`docs/datos/fuentes_y_diccionario.md`](../docs/datos/fuentes_y_diccionario.md)


---
# 1. El problema de negocio y la Pregunta SMART

> *Etapa del flujo: **Pregunta SMART + Hipótesis***

Antes de escribir una sola línea de código, todo análisis exploratorio riguroso debe partir de un **problema de negocio claramente delimitado**.

Ver detalle completo en [`docs/taller1/definicion_problema.md`](../docs/taller1/definicion_problema.md).

## 1.1. Contexto del negocio

El aumento recurrente de casos de **dengue** en el territorio nacional genera **sobrecarga hospitalaria** e **incremento en la mortalidad**, constituyendo una amenaza directa para la salud pública.

Trabajamos con datos de **SIVIGILA** (INS), evento **210 — Dengue**, que registran notificaciones individuales de casos con información epidemiológica, territorial y clínica. El objetivo del análisis es orientar la **vigilancia epidemiológica** y la **respuesta sanitaria territorial** mediante analítica predictiva.

La dirección de salud pública plantea la siguiente inquietud:

> *"¿Qué territorios y periodos concentran mayor riesgo de brotes y fallecimientos por dengue, y qué variables permiten anticiparlos?"*

| Aspecto | Descripción |
|---------|-------------|
| **Tipo de analítica** | **Predictiva** — anticipar territorios y periodos con mayor riesgo |
| **Tipo de problema de IA** | **Regresión** — estimar casos y fallecimientos esperados por municipio |
| **Métricas clave** | Tasa de mortalidad, incidencia de casos, fallecimientos por territorio |
| **Impacto esperado** | Reducir mortalidad, mejorar respuesta sanitaria, ampliar vigilancia en municipios de alto riesgo |

## 1.2. Formulación de la Pregunta SMART

Ver desglose en [`docs/taller1/pregunta_smart.md`](../docs/taller1/pregunta_smart.md).

> **Pregunta SMART:** ¿Puede un sistema de analítica predictiva basado en inteligencia artificial anticipar brotes de dengue con una **precisión superior al 80 %**, permitiendo reducir en al menos **20 %** la tasa de mortalidad y en **30 %** el tiempo de respuesta sanitaria en los territorios priorizados durante un **piloto de 6 meses**, utilizando datos epidemiológicos, climáticos y demográficos del **Instituto Nacional de Salud** y el **IDEAM**?

| Criterio | Cómo se cumple |
|----------|----------------|
| **Específica** | Sistema predictivo con IA, brotes de dengue, datos INS e IDEAM |
| **Medible** | Precisión > 80 % · mortalidad −20 % · respuesta −30 % |
| **Accionable** | Alertas tempranas y priorización territorial |
| **Realista** | Fuentes oficiales (INS, IDEAM, DANE) y técnicas validadas |
| **Temporal** | Piloto de **6 meses** |

## 1.3. Hipótesis de trabajo

A partir del conocimiento del dominio y el estado del arte (ver [`docs/taller1/justificacion_ia.md`](../docs/taller1/justificacion_ia.md)), planteamos hipótesis que el EDA deberá confirmar o refutar:

| ID | Hipótesis | Variables involucradas |
|----|-----------|------------------------|
| **H1** | Existe **estacionalidad**: los casos se concentran en ciertos meses/semanas epidemiológicas | `ANO`, `SEMANA`, `FEC_NOT` |
| **H2** | Hay **heterogeneidad territorial**: algunos departamentos/municipios concentran más casos y fallecimientos | `COD_DPTO_O`, `Departamento_ocurrencia`, `Municipio_ocurrencia` |
| **H3** | Variables climáticas (temperatura, lluvia, humedad) se **asocian positivamente** con la incidencia de brotes | IDEAM + agregación temporal/territorial |
| **H4** | La **hospitalización** y el **perfil demográfico** (edad, sexo) difieren entre territorios de alto y bajo riesgo | `PAC_HOS`, `EDAD`, `SEXO` |
| **H5** | El dataset epidemiológico es **suficiente y de calidad** para alimentar un modelo de regresión con meta de precisión > 80 % | Calidad global del EDA |



---
# 2. Carga del conjunto de datos

> *Etapa del flujo: **Carga de datos***

### Fuentes disponibles

| Fuente | Ubicación | Estado |
|--------|-----------|--------|
| **INS / SIVIGILA** (evento 210) | `data/raw/Datos_YYYY_210.xlsx` (2019–2025) | ✅ Descargado (~872k registros) |
| **IDEAM** (clima) | Pendiente | ⬜ Por integrar |
| **DANE** (demografía) | Pendiente | ⬜ Por integrar |

- **Drive equipo:** [Google Drive — Grupo 3](https://drive.google.com/drive/u/1/folders/1NzXBrdwk3EW74dB6GLmYZ_qp86arPvtH)
- **Utilidades del repo:** `src/config.py`, `src/load_data.py`


In [1]:
# Importación de librerías
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Permite importar desde src/ al ejecutar el notebook
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import DATA_RAW, DATA_PROCESSED
from src.load_data import list_raw_files, load_file, load_all_raw

# Configuración estética
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.titlesize"] = 13
pd.set_option("display.max_columns", None)

print(f"Proyecto: {PROJECT_ROOT}")
print(f"Archivos en data/raw: {len(list_raw_files())}")


Proyecto: /Users/user/Documents/personalProjects/proyecto_analisis_de_datos_I
Archivos en data/raw: 7


In [ ]:
# TODO sección 2): cargar y unificar archivos SIVIGILA 2019-2025

# Opción A — cargar todos los archivos
# datasets = load_all_raw()
# df_list = []
# for year, df_year in datasets.items():
#     df_year = df_year.copy()
#     df_year["archivo_origen"] = year
#     df_list.append(df_year)
# df = pd.concat(df_list, ignore_index=True)

# Opción B — cargar un año para pruebas rápidas
# df = load_file(DATA_RAW / "Datos_2023_210.xlsx")

# Verificaciones mínimas esperadas:
# - df.shape
# - df["COD_EVE"].unique()  # debe ser 210 (dengue)
# - df["Nombre_evento"].unique()

df = None  # reemplazar tras la carga


---
# 3. Diccionario de datos

> *Etapa del flujo: **Diccionario de datos***

El diccionario describe el **significado, tipo y relevancia** de cada variable. Documentación completa en [`docs/datos/fuentes_y_diccionario.md`](../docs/datos/fuentes_y_diccionario.md).

Para cada variable clave documentamos:

- **Variable:** nombre de la columna
- **Tipo:** numérica, categórica, fecha
- **Descripción:** qué representa
- **Relevancia:** objetivo / predictora / contexto / descartar

## 3.1. Diccionario del conjunto de datos SIVIGILA (evento 210)

### Variables objetivo y de desenlace

| Variable | Tipo | Descripción | Relevancia |
|----------|------|-------------|------------|
| `CONSECUTIVE` | int | Identificador único del registro | Clave primaria |
| `confirmados` | int | Caso confirmado (1=Sí, 0=No) | Filtro de casos |
| `CON_FIN` / `Estado_final_de_caso` | int | Condición final del paciente | Desenlace clínico |
| `FEC_DEF` | fecha | Fecha de defunción | **Mortalidad** (100% nulos en muestra — validar) |
| `PAC_HOS` | int | Hospitalizado (1=Sí, 2=No) | Severidad |
| `nom_est_f_caso` | texto | Estado final (lab, probable, etc.) | Clasificación |

### Variables temporales

| Variable | Tipo | Descripción | Relevancia |
|----------|------|-------------|------------|
| `ANO` | int | Año de notificación | Serie temporal |
| `SEMANA` | int | Semana epidemiológica | Agregación temporal |
| `FEC_NOT` | texto/fecha | Fecha de notificación | Convertir a datetime |
| `INI_SIN` | texto/fecha | Inicio de síntomas | Latencia del brote |

### Variables territoriales

| Variable | Tipo | Descripción | Relevancia |
|----------|------|-------------|------------|
| `COD_DPTO_O` / `Departamento_ocurrencia` | int/texto | Departamento de ocurrencia | **Análisis territorial** |
| `COD_MUN_O` / `Municipio_ocurrencia` | int/texto | Municipio de ocurrencia | **Unidad de agregación** |
| `AREA` | int | Urbana / Rural | Contexto |

### Variables demográficas y clínicas

| Variable | Tipo | Descripción | Relevancia |
|----------|------|-------------|------------|
| `EDAD` / `UNI_MED` | int | Edad y unidad (años/meses/días) | Perfil epidemiológico |
| `SEXO` | texto | Sexo (M/F) | Perfil epidemiológico |
| `TIP_CAS` | int | Tipo de caso (2=Probable, 3=Confirmado) | Clasificación |

### Variables pendientes (otras fuentes)

| Fuente | Variables | Estado |
|--------|-----------|--------|
| IDEAM | Temperatura, lluvia, humedad | ⬜ Pendiente integrar |
| DANE | Población, densidad | ⬜ Pendiente integrar |
| Secretarías de Salud | Saneamiento, criaderos | ⬜ Pendiente integrar |


In [ ]:
# TODO sección 3): construir diccionario como DataFrame para visualización

# columnas_clave = [
#     "CONSECUTIVE", "ANO", "SEMANA", "FEC_NOT", "EDAD", "SEXO",
#     "COD_DPTO_O", "Departamento_ocurrencia", "COD_MUN_O", "Municipio_ocurrencia",
#     "confirmados", "PAC_HOS", "CON_FIN", "FEC_DEF", "nom_est_f_caso",
# ]
# diccionario = pd.DataFrame({
#     "variable": columnas_clave,
#     "tipo": [...],
#     "descripcion": [...],
#     "relevancia": [...],
# })
# diccionario


---
# 4. Visión general del conjunto de datos

> *Etapa del flujo: **Visión general (formato, nombres, tipos, unidades, columnas redundantes)***

Antes de entrar al detalle, obtenemos una **fotografía global** del dataset:

- Formato general y número de registros
- Nombres y tipos de columnas
- Valores únicos y cardinalidad
- Columnas redundantes, identificadores o con posible *data leakage*
- Estadísticos descriptivos preliminares

## 4.1. Primeras filas del conjunto de datos


In [ ]:
# TODO sección 4.1)
# df.head(10)


## 4.2. Estructura, tipos de datos y memoria

Revisar si columnas numéricas aparecen como `object` (fechas en texto, códigos mixtos).


In [ ]:
# TODO sección 4.2)
# df.info()
# df.memory_usage(deep=True).sum() / 1e6  # MB aproximados


## 4.3. Dimensiones, columnas y valores únicos

Evaluar cardinalidad de variables categóricas (departamentos, municipios, UPGD).


In [ ]:
# TODO sección 4.3)
# print("Filas:", df.shape[0], "| Columnas:", df.shape[1])
# print("Años:", sorted(df["ANO"].unique()))
# print("Departamentos:", df["Departamento_ocurrencia"].nunique())
# print("Municipios:", df["Municipio_ocurrencia"].nunique())


## 4.4. Estadísticos descriptivos preliminares

**Sobre columnas a descartar o usar con cuidado:**

- **Identificadores:** `CONSECUTIVE` — clave, no predictor
- **Redundantes:** pares código/nombre territorial (`COD_DPTO_O` vs `Departamento_ocurrencia`)
- **100% nulos:** `FEC_DEF`, `GRU_POB`, `CBMTE` — evaluar utilidad
- ***Data leakage*:** variables posteriores al desenlace que no estarían disponibles al predecir


In [ ]:
# TODO sección 4.4)
# df.describe(include="all").T


---
# 5. Limpieza y preparación de los datos

> *Etapa del flujo: preparación previa al análisis (valores perdidos, formato)*

Problemas conocidos del diccionario (ver `docs/datos/fuentes_y_diccionario.md`):

| Aspecto | Hallazgo | Acción sugerida |
|---------|----------|-----------------|
| Fechas en texto | `FEC_NOT`, `INI_SIN`, `FEC_HOS` | Convertir a `datetime` |
| Nulos altos | `FEC_DEF` (100%), `FEC_HOS` (~47%) | Documentar decisión |
| Tipos mixtos | `OCUPACION` | Definir `dtype` al cargar |
| Duplicados | `CONSECUTIVE` único por archivo | Validar al concatenar años |

## 5.1. Identificación de valores perdidos

### Paso 1 — Conteo de nulos por columna


In [ ]:
# TODO sección 5.1)
# nulos = df.isnull().sum()
# nulos[nulos > 0].sort_values(ascending=False)


### Paso 2 — Porcentaje de valores perdidos por columna


In [ ]:
# TODO sección 5.1)
# pct_nulos = (df.isnull().mean() * 100).round(2)
# pct_nulos[pct_nulos > 0].sort_values(ascending=False)


## 5.2. Tratamiento de los valores perdidos

Decidir **columna por columna**: eliminar, imputar (media/moda) o conservar como categoría "desconocido".

| Variable | Faltantes conocidos | Estrategia propuesta | Decisión final |
|----------|---------------------|----------------------|----------------|
| `FEC_HOS` | ~47 % | Conservar nulo si no hospitalizado | _[completar]_ |
| `FEC_DEF` | 100 % | Usar `CON_FIN` / `Estado_final_de_caso` para mortalidad | _[completar]_ |
| `COD_ASE` | ~3.4 % | Moda o categoría "sin EPS" | _[completar]_ |
| `GRU_POB`, `CBMTE` | 100 % | Excluir del análisis | _[completar]_ |

### 5.2.1. Imputación (variables numéricas) — si aplica

### 5.2.2. Imputación por moda (variables categóricas) — si aplica

### 5.2.3. Eliminación de filas — si aplica (ej. registros sin municipio o sin fecha)


In [ ]:
# TODO sección 5.2)
# Ejemplo: filtrar solo casos confirmados
# df_limpio = df[df["confirmados"] == 1].copy()

# Ejemplo: eliminar filas sin municipio de ocurrencia
# df_limpio = df_limpio.dropna(subset=["COD_MUN_O"])

# Documentar cada decisión en markdown antes de ejecutar


### Verificación: estado de valores perdidos tras limpieza


In [ ]:
# TODO sección 5.2)
# df_limpio.isnull().sum().sum()


## 5.3. Corrección del formato de los datos

Convertir fechas, tipos numéricos y estandarizar categorías.


In [ ]:
# TODO sección 5.3)
# cols_fecha = ["FEC_NOT", "INI_SIN", "FEC_HOS", "FEC_CON"]
# for col in cols_fecha:
#     if col in df_limpio.columns:
#         df_limpio[col] = pd.to_datetime(df_limpio[col], dayfirst=True, errors="coerce")


## 5.4. Creación de nuevas columnas (features derivadas)

Ideas para el caso dengue:

| Columna nueva | Fórmula / lógica | Uso |
|---------------|------------------|-----|
| `edad_anios` | Normalizar `EDAD` según `UNI_MED` | Perfil epidemiológico |
| `mes` / `trimestre` | Extraer de `FEC_NOT` | Estacionalidad (H1) |
| `fallecido` | A partir de `CON_FIN` o estado final | Variable de mortalidad |
| `hospitalizado` | `PAC_HOS == 1` | Severidad |
| `depto_mun` | Concatenar código DANE | Clave territorial |

> Cuando se integren IDEAM/DANE: agregar `casos_por_100k_hab`, `tasa_mortalidad`, variables climáticas mensuales por municipio.


In [ ]:
# TODO sección 5.4)
# def edad_en_anios(row):
#     if row["UNI_MED"] == 1:
#         return row["EDAD"]
#     if row["UNI_MED"] == 2:
#         return row["EDAD"] / 12
#     return row["EDAD"] / 365
#
# df_limpio["edad_anios"] = df_limpio.apply(edad_en_anios, axis=1)
# df_limpio["mes"] = df_limpio["FEC_NOT"].dt.month
# df_limpio["hospitalizado"] = df_limpio["PAC_HOS"] == 1


In [ ]:
# TODO sección 5): exportar dataset limpio para el equipo
# DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
# df_limpio.to_parquet(DATA_PROCESSED / "dengue_sivigila_limpio.parquet", index=False)
# print("Exportado:", DATA_PROCESSED / "dengue_sivigila_limpio.parquet")


Con esto concluye la etapa de limpieza. Disponemos de un conjunto **completo, con tipos correctos y variables derivadas** listo para el análisis exploratorio.


---
# 6. Análisis univariado

> *Etapa del flujo: **Análisis univariado***

Comprender la distribución de **cada variable por separado** antes de buscar relaciones.

## 6.1. Variables objetivo: casos, fallecimientos y mortalidad

Iniciamos por las variables que responden a la pregunta de negocio.

Preguntas guía:

- ¿Cómo se distribuyen los casos por año, mes y semana epidemiológica?
- ¿Cuántos fallecimientos y hospitalizaciones hay?
- ¿Cuál es la tasa de mortalidad global y por territorio?


In [ ]:
# TODO sección 6.1)
# casos_por_anio = df_limpio.groupby("ANO").size()
# casos_por_anio.plot(kind="bar", title="Casos de dengue por año")
# plt.ylabel("Número de casos")
# plt.show()


In [ ]:
# TODO sección 6.1) — serie temporal mensual
# casos_mes = df_limpio.groupby([df_limpio["FEC_NOT"].dt.to_period("M")]).size()
# casos_mes.plot(title="Casos por mes")
# plt.show()


**Interpretación (completar tras el análisis):**

- _[Describir tendencia 2019–2025, picos conocidos, estacionalidad]_
- _[Comentar mortalidad y hospitalización]_


## 6.2. Variables numéricas continuas

Variables candidatas: `edad_anios`, `SEMANA`, agregados por territorio.

Incluir histogramas, boxplots y detección de outliers (IQR) si aplica.


In [ ]:
# TODO sección 6.2)
# def analizar_variable_continua(columna, data=df_limpio):
#     fig, axes = plt.subplots(1, 2, figsize=(12, 4))
#     data[columna].hist(bins=30, ax=axes[0])
#     axes[0].set_title(f"Histograma — {columna}")
#     data.boxplot(column=columna, ax=axes[1])
#     axes[1].set_title(f"Boxplot — {columna}")
#     plt.tight_layout()
#     plt.show()
#     display(data[columna].describe())
#
# analizar_variable_continua("edad_anios")


### 6.2.1. Cuantificación de outliers (IQR) — si aplica

> La detección de outliers **no implica** eliminación automática. Documentar y justificar.


In [ ]:
# TODO sección 6.2.1)
# def contar_outliers_iqr(columna, data=df_limpio):
#     q1, q3 = data[columna].quantile([0.25, 0.75])
#     iqr = q3 - q1
#     mask = (data[columna] < q1 - 1.5 * iqr) | (data[columna] > q3 + 1.5 * iqr)
#     return mask.sum(), len(data), mask.sum() / len(data) * 100


## 6.3. Variables categóricas

Variables candidatas: `SEXO`, `PAC_HOS`, `nom_est_f_caso`, `Departamento_ocurrencia`, `AREA`, `TIP_SS`.

Incluir tablas de frecuencia y gráficos de barras.


In [ ]:
# TODO sección 6.3)
# def analizar_variable_categorica(columna, data=df_limpio, top_n=15):
#     freq = data[columna].value_counts().head(top_n)
#     freq.plot(kind="barh", title=f"Frecuencia — {columna}")
#     plt.xlabel("Conteo")
#     plt.show()
#     display(freq.to_frame("conteo"))
#
# analizar_variable_categorica("SEXO")
# analizar_variable_categorica("Departamento_ocurrencia")


**Interpretación (completar tras el análisis):**

- _[Perfil demográfico: sexo, edad]_
- _[Proporción de hospitalizados y tipos de caso]_
- _[Departamentos con mayor conteo absoluto]_


---
# 7. Análisis bivariado

> *Etapa del flujo: **Análisis bivariado***

Exploramos **relaciones entre pares de variables** para contrastar las hipótesis H1–H4.

## 7.1. Continua vs. continua: matriz de correlación

Agregar variables numéricas relevantes (edad, semana, agregados temporales).
Cuando se integre clima: correlacionar temperatura/lluvia/humedad con casos agregados.


In [ ]:
# TODO sección 7.1)
# cols_num = df_limpio.select_dtypes(include="number").columns
# corr = df_limpio[cols_num].corr()
# sns.heatmap(corr, cmap="coolwarm", center=0, annot=False)
# plt.title("Matriz de correlación")
# plt.show()


## 7.2. Continua vs. continua: diagramas de dispersión

Ejemplos: edad vs. hospitalización (codificada), casos mensuales vs. variable climática (cuando esté disponible).


In [ ]:
# TODO sección 7.2)
# sns.scatterplot(data=df_limpio.sample(min(5000, len(df_limpio))), x="edad_anios", y="SEMANA", hue="hospitalizado", alpha=0.3)
# plt.title("Edad vs. semana epidemiológica")
# plt.show()


## 7.3. Categórica vs. continua / conteo: casos por territorio y perfil

Comparar distribución de casos, hospitalizaciones y mortalidad por departamento, sexo, área.


In [ ]:
# TODO sección 7.3)
# top_deptos = df_limpio["Departamento_ocurrencia"].value_counts().head(10).index
# df_top = df_limpio[df_limpio["Departamento_ocurrencia"].isin(top_deptos)]
# sns.countplot(data=df_top, y="Departamento_ocurrencia", order=top_deptos)
# plt.title("Top 10 departamentos por casos")
# plt.show()


In [ ]:
# TODO sección 7.3) — hospitalización por departamento (top 10)
# pd.crosstab(df_top["Departamento_ocurrencia"], df_top["hospitalizado"], normalize="index").plot(kind="bar", stacked=True)
# plt.title("Proporción de hospitalización por departamento")
# plt.show()


## 7.4. Verificación de relación esperada: estacionalidad y brotes (H1)

Analizar si los picos de casos coinciden con periodos específicos del año (mes/semana).


In [ ]:
# TODO sección 7.4)
# casos_por_semana = df_limpio.groupby("SEMANA").size()
# casos_por_semana.plot(title="Casos por semana epidemiológica (todos los años)")
# plt.xlabel("Semana")
# plt.ylabel("Casos")
# plt.show()


## 7.5. Análisis territorial y espacial (opcional, H2)

Mapas de calor o choropleth por departamento/municipio. Requiere coordenadas o shapefiles (pendiente).



In [ ]:
# TODO sección 7.5) — cuando haya geometrías o coordenadas
# import geopandas as gpd  # agregar dependencia si se usa
# casos_depto = df_limpio.groupby("COD_DPTO_O").size().reset_index(name="casos")
# # unir con shapefile de departamentos y graficar mapa


**Interpretación bivariada (completar tras el análisis):**

- _[H1 estacionalidad: confirmada / refutada — evidencia]_
- _[H2 territorial: confirmada / refutada — evidencia]_
- _[H3 clima: pendiente / confirmada — evidencia]_
- _[H4 perfil clínico: confirmada / refutada — evidencia]_


---
# 8. Conclusiones

> *Etapa del flujo: **Conclusiones***

Recordemos la pregunta que orientó todo el análisis:

> **Pregunta SMART:** ¿Puede un sistema de analítica predictiva basado en IA anticipar brotes de dengue con precisión > 80 %, reduciendo ≥ 20 % la mortalidad y ≥ 30 % el tiempo de respuesta sanitaria en un piloto de 6 meses, con datos del INS y el IDEAM?

## 8.1. Contraste de las hipótesis

| Hipótesis | Enunciado | Resultado | Evidencia |
|-----------|---------|-----------|-----------|
| **H1** | Estacionalidad en meses/semanas | _[Confirmada / Refutada / Parcial]_ | _[Sección 6.1, 7.4]_ |
| **H2** | Heterogeneidad territorial | _[Confirmada / Refutada / Parcial]_ | _[Sección 6.3, 7.3, 7.5]_ |
| **H3** | Asociación con variables climáticas | _[Pendiente / Confirmada / Refutada]_ | _[Sección 7.1–7.2]_ |
| **H4** | Perfil clínico difiere por riesgo | _[Confirmada / Refutada / Parcial]_ | _[Sección 6.3, 7.3]_ |
| **H5** | Datos aptos para modelo > 80 % | _[Sí / No / Parcial]_ | _[Calidad EDA global]_ |

## 8.2. Sobre la calidad del conjunto de datos

Completar tras las secciones 4 y 5:

- **Completitud:** _[¿Quedó sin valores críticos faltantes?]_
- **Formato:** _[¿Fechas y tipos corregidos?]_
- **Relevancia:** _[¿Variables predictoras identificadas?]_
- **Tamaño:** _[~872k registros — suficiente para EDA y regresión agregada]_

## 8.3. Limitaciones y recomendaciones

- Integración pendiente de **IDEAM** y **DANE**
- `FEC_DEF` con 100 % nulos — validar otra fuente para mortalidad
- Agregación territorial necesaria antes del modelado
- _[Agregar limitaciones encontradas en el EDA]_

## 8.4. Respuesta a la pregunta de negocio

¿El EDA sugiere que un piloto de 6 meses con precisión > 80 % es **viable** con los datos actuales?

> _[Respuesta argumentada en 1–2 párrafos, vinculando hallazgos del EDA con los indicadores SMART]_


---
# 9. Evidencia de uso de IA generativa

> Requerido por el taller — ver [`docs/taller1/justificacion_ia.md`](../docs/taller1/justificacion_ia.md)

| Pregunta | Respuesta |
|----------|-----------|
| **Herramienta usada** | _[ej. Cursor, ChatGPT, Copilot]_ |
| **Qué partes se hicieron con IA** | _[ej. estructura del notebook, código de carga, gráficos]_ |
| **Qué revisó el equipo** | _[datos, lógica epidemiológica, conclusiones, cifras]_ |
| **Prompts o iteraciones relevantes** | _[opcional — enlazar o resumir]_ |
